In [5]:
from pathlib import Path

import numpy as np
import pandas as pd

from ortools.sat.python import cp_model

In [6]:
DATA_DIR = Path("../data")

PROCESSED_DIR = DATA_DIR / "processed"

In [7]:
recommendations = pd.read_parquet(
    PROCESSED_DIR /
    "candidate_recommendations.parquet"
)

business_features = pd.read_parquet(
    PROCESSED_DIR /
    "business_features.parquet"
)

In [8]:
recommendations = recommendations.merge(
    business_features,
    on="article_id",
    how="left"
)

In [9]:
recommendations["score_norm"] = (
    recommendations["score"]
    - recommendations["score"].min()
) / (
    recommendations["score"].max()
    - recommendations["score"].min()
)

recommendations["margin_norm"] = (
    recommendations["margin_pct"]
    - recommendations["margin_pct"].min()
) / (
    recommendations["margin_pct"].max()
    - recommendations["margin_pct"].min()
)

recommendations["inventory_norm"] = (
    recommendations["inventory"]
    - recommendations["inventory"].min()
) / (
    recommendations["inventory"].max()
    - recommendations["inventory"].min()
)

recommendations["promotion_norm"] = (
    recommendations["promotion_priority"]
    - recommendations["promotion_priority"].min()
) / (
    recommendations["promotion_priority"].max()
    - recommendations["promotion_priority"].min()
)

In [10]:
recommendations["business_score"] = (
      0.60 * recommendations["score_norm"]
    + 0.20 * recommendations["margin_norm"]
    + 0.10 * recommendations["inventory_norm"]
    + 0.10 * recommendations["promotion_norm"]
)

In [11]:
recommendations[
    [
        "article_id",
        "score",
        "margin_pct",
        "inventory",
        "promotion_priority",
        "business_score"
    ]
].head()

,article_id,score,margin_pct,inventory,promotion_priority,business_score
0,806388002,0.239194,0.196081,168,4,0.267140
1,636323001,0.231691,0.667682,220,5,0.456713
2,806388003,0.190779,0.675495,460,5,0.489516
3,741356002,0.187407,0.507172,171,5,0.367107
4,610776001,0.133612,0.495104,185,5,0.335625


In [12]:
def optimize_user_recommendations(
    user_df,
    n_recommendations=10
):
    
    model = cp_model.CpModel()

    n_products = len(user_df)

    x = [
        model.NewBoolVar(
            f"product_{i}"
        )
        for i in range(n_products)
    ]

    scaled_scores = (
        user_df["business_score"] * 1000
    ).astype(int)

    model.Maximize(
        sum(
            scaled_scores.iloc[i] * x[i]
            for i in range(n_products)
        )
    )

    model.Add(
        sum(x) == n_recommendations
    )

    categories = (
        user_df["product_group_name"]
        .fillna("UNKNOWN")
    )

    for category in categories.unique():

        idxs = [
            i
            for i, c in enumerate(categories)
            if c == category
        ]

        model.Add(
            sum(x[i] for i in idxs)
            <= 2
        )

    solver = cp_model.CpSolver()

    status = solver.Solve(model)

    if status not in (
        cp_model.OPTIMAL,
        cp_model.FEASIBLE
    ):
        return pd.DataFrame()

    selected = [
        i
        for i in range(n_products)
        if solver.Value(x[i]) == 1
    ]

    return user_df.iloc[selected]

In [ ]:
# Example for 1 user

test_user = (
    recommendations["customer_id"]
    .iloc[0]
)

user_recs = recommendations[
    recommendations["customer_id"]
    == test_user
]

optimized = optimize_user_recommendations(
    user_recs
)

optimized.head()

,user_idx,item_idx,score,customer_id,article_id,prod_name,product_type_name,product_group_name,colour_group_name,margin_pct,inventory,promotion_priority,score_norm,margin_norm,inventory_norm,promotion_norm,business_score
1,25011,3538,0.231691,00125440be6cd148c3599b9c5a2d55f5838c1b0257d356...,636323001,Skinny H.W Ankle Queens,Trousers,Garment Lower body,Black,0.667682,220,5,0.215982,0.946313,0.378619,1.0,0.456713
2,25011,58333,0.190779,00125440be6cd148c3599b9c5a2d55f5838c1b0257d356...,806388003,Therese tee,T-shirt,Garment Upper body,Beige,0.675495,460,5,0.177225,0.959338,0.913140,1.0,0.489516
8,25011,601,0.112166,00125440be6cd148c3599b9c5a2d55f5838c1b0257d356...,448509014,Perrie Slim Mom Denim TRS,Trousers,Garment Lower body,Blue,0.636457,495,3,0.102754,0.894251,0.991091,0.5,0.389612
23,25011,37491,0.081540,00125440be6cd148c3599b9c5a2d55f5838c1b0257d356...,689109003,Timeless Sports Top,Bikini top,Swimwear,Dark Green,0.692051,238,5,0.073741,0.986942,0.418708,1.0,0.383504
35,25011,5550,0.074727,00125440be6cd148c3599b9c5a2d55f5838c1b0257d356...,600886001,Timeless Highwaist,Swimwear bottom,Swimwear,Black,0.505995,343,5,0.067288,0.676737,0.652561,1.0,0.340976


In [15]:
optimized_results = []

users = recommendations[
    "customer_id"
].unique()

for i, user_id in enumerate(users):

    user_df = recommendations[
        recommendations["customer_id"]
        == user_id
    ]

    selected = (
        optimize_user_recommendations(
            user_df
        )
    )

    optimized_results.append(
        selected
    )

    if (i + 1) % 500 == 0:
        print(
            f"Processed {i+1:,}/{len(users):,}"
        )


Processed 500/5,000
Processed 1,000/5,000
Processed 1,500/5,000
Processed 2,000/5,000
Processed 2,500/5,000
Processed 3,000/5,000
Processed 3,500/5,000
Processed 4,000/5,000
Processed 4,500/5,000
Processed 5,000/5,000


In [16]:
optimized_recommendations = pd.concat(
    optimized_results,
    ignore_index=True
)

In [20]:
pure_recommendations = (
    recommendations
    .sort_values(
        ["customer_id", "score"],
        ascending=[True, False]
    )
    .groupby("customer_id")
    .head(10)
    .copy()
)

In [26]:
pure_recommendations["recommendation_type"] = "ALS"

optimized_recommendations["recommendation_type"] = "BUSINESS_OPTIMIZED"

In [27]:
optimized_recommendations

,user_idx,item_idx,score,customer_id,article_id,prod_name,product_type_name,product_group_name,colour_group_name,margin_pct,inventory,promotion_priority,score_norm,margin_norm,inventory_norm,promotion_norm,business_score,recommendation_type
0,25011,3538,0.231691,00125440be6cd148c3599b9c5a2d55f5838c1b0257d356...,636323001,Skinny H.W Ankle Queens,Trousers,Garment Lower body,Black,0.667682,220,5,0.215982,0.946313,0.378619,1.0,0.456713,BUSINESS_OPTIMIZED
1,25011,58333,0.190779,00125440be6cd148c3599b9c5a2d55f5838c1b0257d356...,806388003,Therese tee,T-shirt,Garment Upper body,Beige,0.675495,460,5,0.177225,0.959338,0.913140,1.0,0.489516,BUSINESS_OPTIMIZED
2,25011,601,0.112166,00125440be6cd148c3599b9c5a2d55f5838c1b0257d356...,448509014,Perrie Slim Mom Denim TRS,Trousers,Garment Lower body,Blue,0.636457,495,3,0.102754,0.894251,0.991091,0.5,0.389612,BUSINESS_OPTIMIZED
3,25011,37491,0.081540,00125440be6cd148c3599b9c5a2d55f5838c1b0257d356...,689109003,Timeless Sports Top,Bikini top,Swimwear,Dark Green,0.692051,238,5,0.073741,0.986942,0.418708,1.0,0.383504,BUSINESS_OPTIMIZED
4,25011,5550,0.074727,00125440be6cd148c3599b9c5a2d55f5838c1b0257d356...,600886001,Timeless Highwaist,Swimwear bottom,Swimwear,Black,0.505995,343,5,0.067288,0.676737,0.652561,1.0,0.340976,BUSINESS_OPTIMIZED
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39975,95008,75012,0.052655,ffe26ec85b801a50021d5c411d97cabc511073099eddc0...,717490057,Cat Tee.,T-shirt,Garment Upper body,Light Orange,0.599960,416,5,0.046378,0.833402,0.815145,1.0,0.376021,BUSINESS_OPTIMIZED
39976,95008,5010,0.050897,ffe26ec85b801a50021d5c411d97cabc511073099eddc0...,465655010,Ginger Thong Maple Low 3p,Underwear bottom,Underwear,Black,0.611528,413,3,0.044712,0.852690,0.808463,0.5,0.328212,BUSINESS_OPTIMIZED
39977,95008,36116,0.048192,ffe26ec85b801a50021d5c411d97cabc511073099eddc0...,719957006,Ginger Ch hipster ctn 3p,Unknown,Unknown,Black,0.588690,141,3,0.042151,0.814612,0.202673,0.5,0.258480,BUSINESS_OPTIMIZED
39978,95008,4792,0.046840,ffe26ec85b801a50021d5c411d97cabc511073099eddc0...,429313006,Charlotte SP Andes,Bra,Underwear,Dark Red,0.633195,495,5,0.040869,0.888814,0.991091,1.0,0.401393,BUSINESS_OPTIMIZED


In [29]:
final_recommendations = pd.concat(
    [
        pure_recommendations,
        optimized_recommendations
    ],
    ignore_index=True
)

In [30]:
final_recommendations.to_parquet(
    PROCESSED_DIR /
    "optimized_recommendations.parquet",
    index=False
)

In [31]:
comparison = (
    final_recommendations
    .groupby("recommendation_type")
    .agg(
        avg_relevance=(
            "score",
            "mean"
        ),
        avg_margin=(
            "margin_pct",
            "mean"
        ),
        avg_inventory=(
            "inventory",
            "mean"
        ),
        products=(
            "article_id",
            "count"
        )
    )
)

comparison

,avg_relevance,avg_margin,avg_inventory,products
recommendation_type,,,,
ALS,0.170937,0.395505,271.176480,50000
BUSINESS_OPTIMIZED,0.101011,0.563299,338.326338,39980
